# Putting it all together

ML is all about experiments. So, we will have codes all over the places.

To put everything all together, we will use scikit learn's ["Pipeline"](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) for that.

From their website:

The purpose of the pipeline is to assemble several steps that can be cross-validated together while setting different parameters. For this, it enables setting parameters of the various steps using their names and the parameter name separated by a '__', as in the example below. A step’s estimator may be replaced entirely by setting the parameter with its name to another estimator, or a transformer removed by setting it to 'passthrough' or None.


We will use the `car sales` dataset to demonstrate the process. Here's our goal:

![pipeline](./img/pipeline.png)

Let's begin...

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor

In [3]:
df = pd.read_csv("./data/car-sales-extended-missing-data.csv")
df

,Make,Colour,Odometer (KM),Doors,Price
0,Honda,White,35431.0,4.0,15323.0
1,BMW,Blue,192714.0,5.0,19943.0
2,Honda,White,84714.0,4.0,28343.0
3,Toyota,White,154365.0,4.0,13434.0
4,Nissan,Blue,181577.0,3.0,14043.0
...,...,...,...,...,...
995,Toyota,Black,35820.0,4.0,32042.0
996,NaN,White,155144.0,3.0,5716.0
997,Nissan,Blue,66604.0,4.0,31570.0
998,Honda,White,215883.0,4.0,4001.0


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Make           951 non-null    object 
 1   Colour         950 non-null    object 
 2   Odometer (KM)  950 non-null    float64
 3   Doors          950 non-null    float64
 4   Price          950 non-null    float64
dtypes: float64(3), object(2)
memory usage: 39.2+ KB


**Observation**: We've got some missing values in all the columns. The `Make` and `Colour` column consists string type data, which we will take care of later aka- turn those into numbers.

Also, We will be trying to predict the `Price`. So, it will be a regression model.

Sneak-peak at the null values:

In [8]:
df.isna().sum()

Make             49
Colour           50
Odometer (KM)    50
Doors            50
Price            50
dtype: int64

Here are our steps:

1. Fill the missing data (Imputer)
2. Convert strings to numbers (encoding -> OneHotEncoder)
3. Build an ML model (RandomForestRegressor)

Everything in one cell $\rightarrow$

In [9]:
# Imports
# Data preprocessing
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Modelling
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV

# To make reproducable numbers
np.random.seed(42)

# Loading Data
df = pd.read_csv("./data/car-sales-extended-missing-data.csv")
df = df.dropna(subset=["Price"])  # Drop missing Prices

# Data Preprocessing
categorical_features = ["Make", "Colour"]
categorical_transformer = Pipeline(
                            steps=[
                                # Handle Missing Values
                                ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
                                # Encoding: Strings -> Numbers
                                ("onehot", OneHotEncoder(handle_unknown="ignore"))
                            ]
                        )

door_feature = ["Doors"]
door_transformer = Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent"))
                    ]
                )

numeric_feature = ["Odometer (KM)"]
numeric_transformer = Pipeline(
                        steps=[
                            ("imputer", SimpleImputer(strategy="mean"))
                        ]
                    )


# Setup the preprocessing steps
preprocessor = ColumnTransformer(
                transformers=[
                    ("catergorical", categorical_transformer, categorical_features),
                    ("door", door_transformer, door_feature),
                    ("numeric", numeric_transformer, numeric_feature)
                ]
            )


# Creating a prerpcessing and modelling pipeline
model = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("model", RandomForestRegressor())
            ]
        )

# Extract Features and lables
X = df.drop("Price", axis=1)
y = df["Price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Fit the model
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('catergorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Make', 'Colour']),
                                                 ('door',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent'))]),
                                                  ['Doors']),
                                                 ('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer())]),
                                                  ['Odometer (KM)'])])),
                ('model', RandomForestRegressor())])

Let's see the score:

In [10]:
model.score(X_test, y_test)

0.22188417408787875

**Interpretation:** The model explains $22\%$ of the variability in the Price with respect to its mean using the selected features.

Let's see the accuracy using MAE:

In [15]:
from sklearn.metrics import mean_absolute_error
y_pred = model.predict(X_train)
mean_absolute_error(y_train, y_pred)

2292.155149620263

**Interpretation:** On average, the Price predictions of our model are off by $\pm 2300$ USD from the actual prices.

We can also to the Hyper Parameter tuning with the pipelines. In this case we will be using GridSearchCV.

In [20]:
# Let's define our grid.
pipe_grid = {
    "preprocessor__numeric__imputer__strategy": ["mean", "median"],
    "model__n_estimators": [50, 100, 500, 1000],
    "model__max_depth": [None, 5],
    "model__max_features": ["auto"],
    "model__min_samples_split": [2, 4]
}

gs_model = GridSearchCV(model, pipe_grid, cv=5, verbose=2)
gs_model.fit(X_train, y_train)

Fitting 5 folds for each of 32 candidates, totalling 160 fits
[CV] END model__max_depth=None, model__max_features=auto, model__min_samples_split=2, model__n_estimators=50, preprocessor__numeric__imputer__strategy=mean; total time=   0.0s
[CV] END model__max_depth=None, model__max_features=auto, model__min_samples_split=2, model__n_estimators=50, preprocessor__numeric__imputer__strategy=mean; total time=   0.0s


C:\ProgramData\anaconda3\lib\site-packages\sklearn\ensemble\_forest.py:413: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features=1.0` or remove this parameter as it is also the default value for RandomForestRegressors and ExtraTreesRegressors.
  warn(
C:\ProgramData\anaconda3\lib\site-packages\sklearn\ensemble\_forest.py:413: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features=1.0` or remove this parameter as it is also the default value for RandomForestRegressors and ExtraTreesRegressors.
  warn(
C:\ProgramData\anaconda3\lib\site-packages\sklearn\ensemble\_forest.py:413: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features=1.0` or remove this parameter as it is also the default value for RandomForestR

[CV] END model__max_depth=None, model__max_features=auto, model__min_samples_split=2, model__n_estimators=50, preprocessor__numeric__imputer__strategy=mean; total time=   0.0s
[CV] END model__max_depth=None, model__max_features=auto, model__min_samples_split=2, model__n_estimators=50, preprocessor__numeric__imputer__strategy=mean; total time=   0.0s
[CV] END model__max_depth=None, model__max_features=auto, model__min_samples_split=2, model__n_estimators=50, preprocessor__numeric__imputer__strategy=mean; total time=   0.0s


C:\ProgramData\anaconda3\lib\site-packages\sklearn\ensemble\_forest.py:413: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features=1.0` or remove this parameter as it is also the default value for RandomForestRegressors and ExtraTreesRegressors.
  warn(
C:\ProgramData\anaconda3\lib\site-packages\sklearn\ensemble\_forest.py:413: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features=1.0` or remove this parameter as it is also the default value for RandomForestRegressors and ExtraTreesRegressors.
  warn(
C:\ProgramData\anaconda3\lib\site-packages\sklearn\ensemble\_forest.py:413: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features=1.0` or remove this parameter as it is also the default value for RandomForestR

[CV] END model__max_depth=None, model__max_features=auto, model__min_samples_split=2, model__n_estimators=50, preprocessor__numeric__imputer__strategy=median; total time=   0.0s
[CV] END model__max_depth=None, model__max_features=auto, model__min_samples_split=2, model__n_estimators=50, preprocessor__numeric__imputer__strategy=median; total time=   0.0s
[CV] END model__max_depth=None, model__max_features=auto, model__min_samples_split=2, model__n_estimators=50, preprocessor__numeric__imputer__strategy=median; total time=   0.0s


...

[CV] END model__max_depth=5, model__max_features=auto, model__min_samples_split=4, model__n_estimators=1000, preprocessor__numeric__imputer__strategy=median; total time=   1.0s


C:\ProgramData\anaconda3\lib\site-packages\sklearn\ensemble\_forest.py:413: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features=1.0` or remove this parameter as it is also the default value for RandomForestRegressors and ExtraTreesRegressors.
  warn(


[CV] END model__max_depth=5, model__max_features=auto, model__min_samples_split=4, model__n_estimators=1000, preprocessor__numeric__imputer__strategy=median; total time=   1.0s


C:\ProgramData\anaconda3\lib\site-packages\sklearn\ensemble\_forest.py:413: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features=1.0` or remove this parameter as it is also the default value for RandomForestRegressors and ExtraTreesRegressors.
  warn(


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('catergorical',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(fill_value='missing',
                                                                                                        strategy='constant')),
                                                                                         ('onehot',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['Make',
                                                                          'Colour']),
                                                                        ('door',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent'))]),
                                                                         ['Doors']),
                                                                        ('numeric',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer())]),
                                                                         ['Odometer '
                                                                          '(KM)'])])),
                                       ('model', RandomForestRegressor())]),
             param_grid={'model__max_depth': [None, 5],
                         'model__max_features': ['auto'],
                         'model__min_samples_split': [2, 4],
                         'model__n_estimators': [50, 100, 500, 1000],
                         'preprocessor__numeric__imputer__strategy': ['mean',
                                                                      'median']},
             verbose=2)

Well that's a lot of underscores. Let's look at the first key. We are actually saying that, try our imputer strategies for our `numerical_feature` aka "Odometer (KM)" with "mean" and "median" and see what's better. [Here's](https://youtu.be/ce3H68KvmUc?si=CB7NaZ4u373YStU9&t=126) a breakdown.

Let's check the Score now.

In [21]:
gs_model.score(X_test, y_test)

0.3327227086526432

A bit improvement

In [23]:
gs_model.best_params_

{'model__max_depth': 5,
 'model__max_features': 'auto',
 'model__min_samples_split': 2,
 'model__n_estimators': 50,
 'preprocessor__numeric__imputer__strategy': 'median'}